In [9]:
%pip install -q -U ir_measures sentence_transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.7/760.7 kB 20.1 MB/s eta 0:00:0000:01


In [1]:
import os

os.environ["HF_TOKEN"] = input()

In [2]:
from datasets import load_dataset, Dataset
import os, json, time, random, pathlib
import requests
import numpy as np
import pandas as pd
import torch
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator

/tmp/ipykernel_2724/3150954647.py:12: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator


In [3]:
#dataset = load_dataset("pameydorke/arcanum-cross-platform-queries-synthetic")
dataset = load_dataset("pameydorke/arcanumchat-cross-platform-with-vague-queries-qwen-36-a3b-fp8")

In [4]:
traind_df = dataset['train'].to_pandas()
val_df = dataset['validation'].to_pandas()
test_df = dataset['test'].to_pandas()
full_df = pd.concat([traind_df, val_df, test_df])

unique_docs_df = full_df.drop_duplicates(subset=['text'])
unique_docs_df = unique_docs_df.reset_index()
unique_docs_df = unique_docs_df.drop(columns=['query'])

unique_docs_ds = Dataset.from_pandas(unique_docs_df)

unique_docs_ds

Dataset({
    features: ['index', 'doc_id', 'link', 'title', 'text'],
    num_rows: 1151
})

In [5]:
DATA_URL = "https://raw.githubusercontent.com/ankorn/arcanumsearch/refs/heads/main/data/retriever_test_ds.json"
test_ds = requests.get(DATA_URL, timeout=30).json()

In [6]:
BASE_RETRIEVER_MODEL_ID = "BAAI/bge-m3"
FT_RETRIEVER_MODEL_ID = "pameydorke/arcanumchat-retriever-bge-m3"

BASE_RERANKER_MODEL_ID = "BAAI/bge-reranker-v2-m3"
FT_RERANKER_MODEL_ID = "pameydorke/arcanumchat-reranker-bge-reranker-v2-m3"

In [7]:

base_retriever_model = SentenceTransformer(BASE_RETRIEVER_MODEL_ID)
ft_retriever_model = SentenceTransformer(FT_RETRIEVER_MODEL_ID)

base_reranker_model = CrossEncoder(BASE_RERANKER_MODEL_ID, num_labels=1)
ft_reranker_model = CrossEncoder(FT_RERANKER_MODEL_ID, num_labels=1)

base_retriever_model.eval()
ft_retriever_model.eval()
base_reranker_model.eval()
ft_reranker_model.eval()

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/97.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

CrossEncoder(
  (0): Transformer({'transformer_task': 'sequence-classification', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'logits'}}, 'module_output_name': 'scores', 'architecture': 'XLMRobertaForSequenceClassification'})
)

In [8]:
queries = {}
corpus = {}
relevant_docs = {}
url2text = {}

unique_docs = {}
for idx, example in enumerate(unique_docs_ds):
    doc_link = example["link"]
    doc_text = example["text"]
    url2text[doc_link] = doc_text

    if doc_text not in unique_docs:
        doc_id = example["doc_id"]
        unique_docs[doc_text] = doc_id
        corpus[doc_id] = doc_text

skipped_counter = 0
vague_counter = 0
for idx, example in enumerate(test_ds):
    # if example['is_vague']:
    #     vague_counter += 1
    #     continue

    query = example['query']
    url = example['url']

    if 'https://www.reddit.com' in url:
        url = url.replace('https://www.reddit.com', 'https://reddit.com')

    if url not in url2text:
        skipped_counter += 1
        continue

    doc_text = url2text[url]

    if doc_text not in unique_docs:
        skipped_counter += 1
        continue

    doc_id = unique_docs[doc_text]

    query_id = f"query_{idx}"
    queries[query_id] = query
    relevant_docs[query_id] = {doc_id}

print("Skipped:", skipped_counter)
print("Skipped vague:", vague_counter)

Skipped: 2
Skipped vague: 0


In [ ]:
base_retriever_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="arcanum-retriever-test",
)
ft_retriever_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="arcanum-retriever-test",
)

base_retriever_results = base_retriever_evaluator(base_retriever_model)
ft_retriever_results = ft_retriever_evaluator(ft_retriever_model)

In [11]:
base_retriever_results

{'arcanum-retriever-test_cosine_accuracy@1': 0.6578947368421053,
 'arcanum-retriever-test_cosine_accuracy@3': 0.7842105263157895,
 'arcanum-retriever-test_cosine_accuracy@5': 0.8473684210526315,
 'arcanum-retriever-test_cosine_accuracy@10': 0.868421052631579,
 'arcanum-retriever-test_cosine_precision@1': 0.6578947368421053,
 'arcanum-retriever-test_cosine_precision@3': 0.26140350877192975,
 'arcanum-retriever-test_cosine_precision@5': 0.16947368421052625,
 'arcanum-retriever-test_cosine_precision@10': 0.08684210526315787,
 'arcanum-retriever-test_cosine_recall@1': 0.6578947368421053,
 'arcanum-retriever-test_cosine_recall@3': 0.7842105263157895,
 'arcanum-retriever-test_cosine_recall@5': 0.8473684210526315,
 'arcanum-retriever-test_cosine_recall@10': 0.868421052631579,
 'arcanum-retriever-test_cosine_ndcg@10': 0.7664273704408339,
 'arcanum-retriever-test_cosine_mrr@10': 0.7331766917293234,
 'arcanum-retriever-test_cosine_map@100': 0.7376623485506122}

In [12]:
ft_retriever_results

{'arcanum-retriever-test_cosine_accuracy@1': 0.6105263157894737,
 'arcanum-retriever-test_cosine_accuracy@3': 0.8,
 'arcanum-retriever-test_cosine_accuracy@5': 0.8736842105263158,
 'arcanum-retriever-test_cosine_accuracy@10': 0.9263157894736842,
 'arcanum-retriever-test_cosine_precision@1': 0.6105263157894737,
 'arcanum-retriever-test_cosine_precision@3': 0.2666666666666666,
 'arcanum-retriever-test_cosine_precision@5': 0.1747368421052631,
 'arcanum-retriever-test_cosine_precision@10': 0.09263157894736841,
 'arcanum-retriever-test_cosine_recall@1': 0.6105263157894737,
 'arcanum-retriever-test_cosine_recall@3': 0.8,
 'arcanum-retriever-test_cosine_recall@5': 0.8736842105263158,
 'arcanum-retriever-test_cosine_recall@10': 0.9263157894736842,
 'arcanum-retriever-test_cosine_ndcg@10': 0.7667290322679688,
 'arcanum-retriever-test_cosine_mrr@10': 0.7157602339181285,
 'arcanum-retriever-test_cosine_map@100': 0.7193342021850796}

In [ ]:
# base_reranker_evaluator = CrossEncoderRerankingEvaluator(
#     samples=hard_eval_samples,
#     at_k=10,
#     name="val-hard-negatives",
#     batch_size=64,
#     show_progress_bar=True,
#     always_rerank_positives=True,
# )
# ft_reranker_evaluator = CrossEncoderRerankingEvaluator(
#     samples=hard_eval_samples,
#     at_k=10,
#     name="val-hard-negatives",
#     batch_size=64,
#     show_progress_bar=True,
#     always_rerank_positives=True,
# )

# base_reranker_results = base_reranker_evaluator(base_reranker_model)
# ft_reranker_results = ft_reranker_evaluator(ft_reranker_model)

In [17]:
import math
import torch
import numpy as np
import pandas as pd
from sentence_transformers import util


def encode_queries_and_corpus(model, queries, corpus, batch_size=64):
    qids = list(queries.keys())
    cids = list(corpus.keys())

    q_texts = [queries[q] for q in qids]
    c_texts = [corpus[c] for c in cids]

    q_emb = model.encode(
        q_texts,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    c_emb = model.encode(
        c_texts,
        batch_size=batch_size,
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return qids, cids, q_emb, c_emb

In [ ]:
# def per_query_metrics(model, queries, corpus, relevant_docs, k_values=(1, 5, 10)):
#     qids, cids, q_emb, c_emb = encode_queries_and_corpus(model, queries, corpus)

#     # (num_queries, num_corpus)
#     sim = util.cos_sim(q_emb, c_emb)

#     results = {}

#     for i, qid in enumerate(qids):
#         scores = sim[i]
#         ranked_idx = torch.argsort(scores, descending=True).tolist()
#         ranked_cids = [cids[j] for j in ranked_idx]

#         rel = relevant_docs.get(qid, set())
#         metrics = {}

#         for k in k_values:
#             topk = ranked_cids[:k]

#             # recall@k
#             hits = sum(1 for d in topk if d in rel)
#             metrics[f"recall@{k}"] = hits / len(rel) if rel else 0.0

#             # ndcg@k
#             dcg = 0.0
#             for rank, d in enumerate(topk, start=1):
#                 if d in rel:
#                     dcg += 1.0 / math.log2(rank + 1)

#             idcg = sum(
#                 1.0 / math.log2(r + 1)
#                 for r in range(1, min(len(rel), k) + 1)
#             )
#             metrics[f"ndcg@{k}"] = dcg / idcg if idcg > 0 else 0.0

#         # MRR
#         rr = 0.0
#         for rank, d in enumerate(ranked_cids, start=1):
#             if d in rel:
#                 rr = 1.0 / rank
#                 break
#         metrics["mrr"] = rr

#         results[qid] = metrics

#     return results

In [40]:
import torch
import numpy as np
from sentence_transformers import util
import ir_measures
from ir_measures import nDCG, RR, Recall

def per_query_metrics(model, reranker, queries, corpus, relevant_docs, k_values=(1, 3, 5, 10), k_retrieval=10):
    qids, cids, q_emb, c_emb = encode_queries_and_corpus(model, queries, corpus)

    sim = util.cos_sim(q_emb, c_emb)
    initial_ranked_indices = torch.argsort(sim, dim=1, descending=True)[:, :k_retrieval]
    initial_ranked_cids = [[cids[i] for i in row] for row in initial_ranked_indices.tolist()]

    reranked_cids = []
    for i, query_text in enumerate(queries):
        candidate_cids = initial_ranked_cids[i]
        candidate_docs = [corpus[cid] for cid in candidate_cids]

        pairs = [(query_text, doc) for doc in candidate_docs]
        scores = reranker.predict(pairs, convert_to_numpy=True)
        order = np.argsort(scores)[::-1]
        reranked_cids.append([candidate_cids[j] for j in order])

    # Build qrels and run dictionaries for ir_measures
    qrels = {qid: {doc_id: 1 for doc_id in rel_set} for qid, rel_set in relevant_docs.items()}
    run = {
        qid: {doc_id: float(len(reranked_cids[i]) - rank) for rank, doc_id in enumerate(reranked_cids[i])}
        for i, qid in enumerate(qids)
    }

    measures = []
    for k in k_values:
        measures.extend([Recall @ k, nDCG @ k])
    measures.append(RR)  # RR gives reciprocal rank per query; averaging yields MRR

    results = {qid: {} for qid in qids}
    for metric in ir_measures.iter_calc(measures, qrels, run):
        qid = metric.query_id
        name = str(metric.measure)
        if name == "RR":
            name = "mrr"  # Per-query RR is the MRR contribution
        results[qid][name] = metric.value

    return results

In [ ]:
base_per_q = per_query_metrics(base_retriever_model, base_reranker_model, queries, corpus, relevant_docs, k_retrieval=20)

In [ ]:
ft_per_q = per_query_metrics(ft_retriever_model, ft_reranker_model, queries, corpus, relevant_docs, k_retrieval=20)

In [ ]:
recall_metrics = ["R@3", "R@5", "R@10"]
ndcg_metrics = ["nDCG@10"]

def print_metrics(metrics):
    for metric in metrics:
        base = np.array([base_per_q[q][metric] for q in queries])
        ft   = np.array([ft_per_q[q][metric]   for q in queries])
        delta = ft - base

        print("base mean:", base.mean())
        print("ft mean:  ", ft.mean())
        # print("mean delta:", delta.mean())
        print("improved:", (delta > 0).sum())
        print("worse:   ", (delta < 0).sum())
        print("same:    ", (delta == 0).sum())

        rng = np.random.default_rng(0)
        n = len(delta)

        boot = np.array([
            delta[rng.integers(0, n, n)].mean()
            for _ in range(10_000)
        ])

        lo, hi = np.percentile(boot, [2.5, 97.5])
        print(f"mean delta = {delta.mean():.4f}")
        print(f"95% CI     = [{lo:.4f}, {hi:.4f}]", "\n")

In [ ]:
# k_retrieve = 10

print_metrics(recall_metrics)

base mean: 0.18421052631578946
ft mean:   0.2578947368421053
improved: 41
worse:    27
same:     122
mean delta = 0.0737
95% CI     = [-0.0105, 0.1579] 

base mean: 0.41578947368421054
ft mean:   0.4789473684210526
improved: 52
worse:    40
same:     98
mean delta = 0.0632
95% CI     = [-0.0368, 0.1632] 

base mean: 0.868421052631579
ft mean:   0.9263157894736842
improved: 17
worse:    6
same:     167
mean delta = 0.0579
95% CI     = [0.0105, 0.1053] 



In [ ]:
# k_retrieve = 10

print_metrics(ndcg_metrics)

base mean: 0.3769514890641302
ft mean:   0.41294726989330655
improved: 90
worse:    76
same:     24
mean delta = 0.0360
95% CI     = [-0.0100, 0.0814] 



In [44]:
# k_retrieve = 20

print_metrics(recall_metrics)

base mean: 0.12631578947368421
ft mean:   0.12105263157894737
improved: 18
worse:    19
same:     153
mean delta = -0.0053
95% CI     = [-0.0684, 0.0579] 

base mean: 0.17894736842105263
ft mean:   0.21052631578947367
improved: 31
worse:    25
same:     134
mean delta = 0.0316
95% CI     = [-0.0474, 0.1105] 

base mean: 0.3736842105263158
ft mean:   0.4789473684210526
improved: 54
worse:    34
same:     102
mean delta = 0.1053
95% CI     = [0.0105, 0.2000] 



In [45]:
# k_retrieve = 20

print_metrics(ndcg_metrics)

base mean: 0.17542774503064745
ft mean:   0.21110015252698672
improved: 72
worse:    50
same:     68
mean delta = 0.0357
95% CI     = [-0.0135, 0.0848] 



In [36]:
rows = []
for qid in queries:
    row = {"qid": qid, "query": queries[qid]}
    for k, v in base_per_q[qid].items():
        row[f"base_{k}"] = v
    for k, v in ft_per_q[qid].items():
        row[f"ft_{k}"] = v
    rows.append(row)

df = pd.DataFrame(rows)
df["delta_ndcg@10"] = df["ft_nDCG@10"] - df["base_nDCG@10"]
df = df.sort_values("delta_ndcg@10")

# Queries where fine-tuning hurt most
print(df[["qid", "query", "base_nDCG@10", "ft_nDCG@10", "delta_ndcg@10"]].head(20))

           qid                                  query  base_nDCG@10  \
61    query_63                        ancesctral swrd      1.000000   
36    query_36                                    UAP      1.000000   
13    query_13  adventure teammate with a elefant gun      1.000000   
125  query_127                        character diary      1.000000   
97    query_99                       gambling mastrer      1.000000   
126  query_128                               timeline      1.000000   
173  query_175                            Zefir blimp      1.000000   
0      query_0                  Jaina stopped healing      1.000000   
41    query_41                        flickering icon      1.000000   
47    query_47                       intolerqnce book      1.000000   
46    query_46      psychological book on intolerance      1.000000   
38    query_38                                   tula      1.000000   
165  query_167         ruby place near grey mountains      1.000000   
60    

In [38]:
# Build a list of (qid, base, ft, delta) for recall@5
rows = []
for qid in queries:
    b = base_per_q[qid]["R@10"]
    f = ft_per_q[qid]["R@10"]
    rows.append((qid, b, f, f - b))

# Keep only the ones that got worse
worse = [r for r in rows if r[3] < 0]
worse.sort(key=lambda r: r[3])  # most negative first

print(f"Total worse on recall@10: {len(worse)}")
print("=" * 80)

for qid, b, f, d in worse:
    # print(f"qid:     {qid}")
    print(f"query:   {queries[qid]}")
    # print(f"base:    {b:.3f}   ft: {f:.3f}   delta: {d:+.3f}")

    rel_docs = relevant_docs[qid]
    # print(f"relevant doc ids: {rel_docs}")

    # Show the text of the relevant docs (truncated)
    for doc_id in rel_docs:
        text = corpus[doc_id]
        # print(f"  [{doc_id}] {text[:300]}...")

    # If you want to see where the fine-tuned model ranked the gold doc,
    # you need the ranked list. See helper below.
    print("-" * 80)

Total worse on recall@10: 6
query:   Arcane Sword dungeon coordinates
--------------------------------------------------------------------------------
query:   find elves
--------------------------------------------------------------------------------
query:   UAP
--------------------------------------------------------------------------------
query:   ancesctral swrd
--------------------------------------------------------------------------------
query:   dogde
--------------------------------------------------------------------------------
query:   sniper rifgle schema
--------------------------------------------------------------------------------
